# Qwen3-8B Full-Parameter Fine-Tuning Verification

This notebook verifies the fine-tuning capability of the **Ascend 910B CANN image** by running full-parameter SFT fine-tuning for Qwen3-8B with MindSpeed-LLM.

**Workflow:**
1. Environment check
2. Prepare a sample dataset (Alpaca format)
3. Clone the MindSpeed-LLM scripts
4. Convert HF weights to Megatron weights
5. Preprocess the data
6. Start fine-tuning
7. Run inference validation

> The training parameters are set for verification mode (few iterations + short sequence length). Increase `TRAIN_ITERS` and `SEQ_LENGTH` for production use.

## 0. Parameter Configuration

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=ImportWarning)
warnings.filterwarnings('ignore', category=UserWarning)

from pathlib import Path

# ===== Path configuration =====
HF_MODEL_DIR = Path('/opt/app-root/src/models/Qwen3-8B')
WORK_DIR = Path('/opt/app-root/src/Qwen3-8B-work-dir')
MINDSPEED_LLM_DIR = WORK_DIR / 'MindSpeed-LLM'
DATA_DIR = WORK_DIR / 'finetune_dataset'
RAW_DATA_FILE = DATA_DIR / 'alpaca_sample.jsonl'
PROCESSED_DATA_PREFIX = DATA_DIR / 'alpaca'
OUTPUT_DIR = WORK_DIR / 'output' / 'qwen3_8b_finetuned'
LOGS_DIR = WORK_DIR / 'logs'

# ===== Optional: real dataset path =====
ALPACA_PARQUET = Path('/opt/app-root/src/datasets/alpaca/train-00000-of-00001-a09b74b3ef9c3b56.parquet')

# ===== Ascend environment scripts =====
CANN_ENV = '/usr/local/Ascend/cann/set_env.sh'
ATB_ENV = '/usr/local/Ascend/nnal/atb/set_env.sh'

# ===== Parallelism configuration (must match weight conversion) =====
TP = 2   # With TP=1, one card holds about 4.1B parameters; fp32 gradient buffers + bf16 weights require about 30 GiB, exceeding the 910B 29 GiB memory limit
PP = 2   # At least TPxPP=4 NPUs are required; for a single card, set TP=1 and PP=1 (OOM is possible)

# ===== Weight conversion output (path includes parallel settings to avoid reusing stale weights after TP/PP changes) =====
MCORE_WEIGHTS_DIR = WORK_DIR / 'model_weights' / f'qwen3_mcore_tp{TP}_pp{PP}'

# ===== Training hyperparameters (verification mode) =====
SEQ_LENGTH = 512     # 4096 is recommended for production
TRAIN_ITERS = 50     # 2000+ is recommended for production
MBS = 1
LR = 1.25e-6
MIN_LR = 1.25e-7

# ===== Data preprocessing =====
HANDLER_NAME = 'AlpacaStyleInstructionHandler'
TOKENIZER_TYPE = 'PretrainedFromHF'
PROMPT_TYPE = 'qwen3'
ENABLE_THINKING = 'none'

print('Configuration loaded')
print(f'  Model: {HF_MODEL_DIR}')
print(f'  Dataset: {ALPACA_PARQUET}' if ALPACA_PARQUET.exists() else '  Dataset: not found, using built-in sample data')
print(f'  TP={TP}, PP={PP}, SEQ={SEQ_LENGTH}, ITERS={TRAIN_ITERS}')

## Helper Function

In [ ]:
import os
import subprocess

_SUPPRESS_WARNINGS = 'ignore::DeprecationWarning,ignore::ImportWarning,ignore::UserWarning'

def run_cmd(cmd, cwd=None, check=True):
    'Run a bash command in the Ascend environment and stream output in real time'
    env_prefix = f'source {CANN_ENV} && source {ATB_ENV}'
    full_cmd = f'{env_prefix} && {cmd}'
    print(f'$ {cmd}\n')
    run_env = os.environ.copy()
    run_env['PYTHONWARNINGS'] = _SUPPRESS_WARNINGS
    result = subprocess.run(
        ['bash', '-lc', full_cmd],
        cwd=str(cwd or WORK_DIR),
        text=True,
        env=run_env,
    )
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed with return code: {result.returncode}')
    return result

print('Helper function defined: run_cmd()')

## 1. Environment Check

In [ ]:
import warnings
with warnings.catch_warnings():
    warnings.simplefilter('ignore', DeprecationWarning)
    warnings.simplefilter('ignore', ImportWarning)
    warnings.simplefilter('ignore', UserWarning)
    import torch
    import torch_npu

print('=' * 60)
print('Environment Check')
print('=' * 60)

# PyTorch & NPU
print(f'PyTorch:    {torch.__version__}')
print(f'torch_npu:  {torch_npu.__version__}')
nproc = torch.npu.device_count()
print(f'NPU count:  {nproc}')
for i in range(nproc):
    print(f'  NPU {i}: {torch.npu.get_device_name(i)}')

# MindSpeed
with warnings.catch_warnings():
    warnings.simplefilter('ignore', DeprecationWarning)
    warnings.simplefilter('ignore', ImportWarning)
    warnings.simplefilter('ignore', UserWarning)
    import mindspeed
    import mindspeed_llm
print('MindSpeed:     installed')
print('MindSpeed-LLM: installed')

# Model files
print(f'\nModel directory: {HF_MODEL_DIR}')
assert HF_MODEL_DIR.exists(), f'Model directory does not exist: {HF_MODEL_DIR}'
model_files = sorted(HF_MODEL_DIR.glob('*'))
for f in model_files[:5]:
    if f.is_file():
        print(f'  {f.name} ({f.stat().st_size / 1e9:.2f} GB)')
if len(model_files) > 5:
    print(f'  ... {len(model_files)} files in total')

# Parallelism validation
assert nproc >= TP * PP, f'NPU count ({nproc}) < TP*PP ({TP*PP}); reduce PP'
DP = nproc // (TP * PP)
GBS = DP * MBS
print(f'\nParallelism: TP={TP}, PP={PP}, DP={DP}, GBS={GBS}')
assert torch.npu.is_available(), 'NPU is not available'
print('\nEnvironment check passed!')

## 2. Prepare a Sample Dataset

Create sample data in Alpaca format to verify the fine-tuning workflow.

To use a real dataset, place a JSONL file at `RAW_DATA_FILE`, with one JSON object per line:
```json
{"instruction": "...", "input": "...", "output": "..."}
```

In [ ]:
import json
import warnings
import pandas as pd

DATA_DIR.mkdir(parents=True, exist_ok=True)

if ALPACA_PARQUET.exists():
    print(f'Loading Alpaca dataset: {ALPACA_PARQUET.name}')
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', DeprecationWarning)
        df = pd.read_parquet(ALPACA_PARQUET)
    print(f'{len(df)} samples loaded, columns: {list(df.columns)}')

    # Convert to JSONL (instruction / input / output)
    with open(RAW_DATA_FILE, 'w', encoding='utf-8') as f:
        for item in df[['instruction', 'input', 'output']].to_dict('records'):
            item['input'] = item.get('input') or ''
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

    print(f'Converted to JSONL: {RAW_DATA_FILE}')
    print('\nSample records:')
    for item in df[['instruction', 'input', 'output']].head(3).to_dict('records'):
        inp = f' {item["input"]}' if item['input'] else ''
        print(f'  Q: {item["instruction"][:80]}{inp[:40]}')
        print(f'  A: {str(item["output"])[:80]}')
else:
    print('Alpaca dataset not found, using built-in sample data\n')
    sample_data = [
        {'instruction': 'Translate the following sentence into French', 'input': 'The weather is nice today.', 'output': "Il fait beau aujourd'hui."},
        {'instruction': 'Translate the following sentence into Spanish', 'input': 'I like programming.', 'output': 'Me gusta programar.'},
        {'instruction': 'Summarize the sentence in one short phrase', 'input': 'Machine learning is fascinating and widely used in many fields.', 'output': 'Machine learning is broadly useful.'},
        {'instruction': 'Rewrite the sentence in a more formal tone', 'input': 'Hello, how are you?', 'output': 'Hello, how are you doing today?'},
        {'instruction': 'Introduce Python in one sentence', 'input': '', 'output': 'Python is a high-level general-purpose programming language known for its readability and rich ecosystem.'},
        {'instruction': 'List three common sorting algorithms', 'input': '', 'output': 'Three common sorting algorithms are bubble sort, quicksort, and merge sort.'},
        {'instruction': 'Explain what deep learning is', 'input': '', 'output': 'Deep learning is a branch of machine learning that uses multi-layer neural networks to learn hierarchical representations of data.'},
        {'instruction': 'Write a Python function to add two numbers', 'input': '', 'output': 'def add(a, b):\n    return a + b'},
        {'instruction': 'Rewrite the sentence to be more concise', 'input': 'Artificial intelligence is changing the world.', 'output': 'AI is transforming the world.'},
        {'instruction': 'What is a GPU?', 'input': '', 'output': 'A GPU is a graphics processing unit designed to accelerate highly parallel computation, especially for training and inference workloads.'},
    ]
    with open(RAW_DATA_FILE, 'w', encoding='utf-8') as f:
        for item in sample_data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    print(f'Sample dataset created: {RAW_DATA_FILE}')
    print(f'{len(sample_data)} samples in total')

## 3. Clone MindSpeed-LLM

The `mindspeed_llm` Python package is already installed in the image, but the training scripts (`convert_ckpt_v2.py`, `preprocess_data.py`, `posttrain_gpt.py`, and others) must be run from the repository directory.

In [ ]:
if MINDSPEED_LLM_DIR.exists():
    print(f'Already exists: {MINDSPEED_LLM_DIR}')
else:
    print('Cloning MindSpeed-LLM (shallow clone)...')
    run_cmd(f'git clone --depth 1 https://gitcode.com/ascend/MindSpeed-LLM.git {MINDSPEED_LLM_DIR}')

# Validate required scripts
scripts = [
    ('Weight conversion', 'convert_ckpt_v2.py'),
    ('Data preprocessing', 'preprocess_data.py'),
    ('Fine-tuning', 'posttrain_gpt.py'),
    ('Inference', 'inference.py'),
]
for name, script in scripts:
    exists = (MINDSPEED_LLM_DIR / script).exists()
    print(f'  [{name}] {script}: {"OK" if exists else "MISSING"}')

assert all((MINDSPEED_LLM_DIR / s).exists() for _, s in scripts), 'Required scripts are missing'
print('\nScript check passed!')

## 4. HF Weight to Megatron Weight Conversion

Convert HuggingFace-format weights to Megatron format, split by TP/PP. The first conversion usually takes about 5-10 minutes.

In [ ]:
MCORE_WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

# Check whether conversion has already been completed
converted = any(MCORE_WEIGHTS_DIR.glob('iter_*'))

if converted:
    print(f'Weights already exist, skipping conversion: {MCORE_WEIGHTS_DIR}')
    for p in sorted(MCORE_WEIGHTS_DIR.iterdir()):
        print(f'  {p.name}')
else:
    convert_cmd = ' && '.join([
        f'cd {MINDSPEED_LLM_DIR}',
        f'python convert_ckpt_v2.py'
        ' --load-model-type hf'
        ' --save-model-type mg'
        f' --target-tensor-parallel-size {TP}'
        f' --target-pipeline-parallel-size {PP}'
        f' --load-dir {HF_MODEL_DIR}'
        f' --save-dir {MCORE_WEIGHTS_DIR}'
        ' --model-type-hf qwen3',
    ])
    print('Running weight conversion (about 5-10 minutes)...')
    run_cmd(convert_cmd, cwd=MINDSPEED_LLM_DIR)
    print('Weight conversion completed!')
    for p in sorted(MCORE_WEIGHTS_DIR.iterdir()):
        print(f'  {p.name}')

## 5. Data Preprocessing

Convert Alpaca-format JSONL data into the binary format required by MindSpeed-LLM training.

In [ ]:
preprocess_cmd = ' && '.join([
    f'cd {MINDSPEED_LLM_DIR}',
    f'python preprocess_data.py'
    f' --input {RAW_DATA_FILE}'
    f' --tokenizer-name-or-path {HF_MODEL_DIR}'
    f' --output-prefix {PROCESSED_DATA_PREFIX}'
    f' --handler-name {HANDLER_NAME}'
    f' --tokenizer-type {TOKENIZER_TYPE}'
    ' --workers 4'
    ' --log-interval 1'
    f' --enable-thinking {ENABLE_THINKING}'
    f' --prompt-type {PROMPT_TYPE}',
])

print('Running data preprocessing...')
run_cmd(preprocess_cmd, cwd=MINDSPEED_LLM_DIR)

# Verify outputs
print('\nPreprocessing outputs:')
for f in sorted(PROCESSED_DATA_PREFIX.parent.glob('alpaca*')):
    print(f'  {f.name} ({f.stat().st_size / 1024:.1f} KB)')
print('Data preprocessing completed!')

## 6. Start Fine-Tuning

Run full-parameter SFT fine-tuning with MindSpeed-LLM. Training logs are streamed to the notebook in real time.

> In verification mode, `TRAIN_ITERS=50`. For a full fine-tuning run, 2000+ iterations are recommended.

In [ ]:
import torch

nproc = torch.npu.device_count()
DP = nproc // (TP * PP)
GBS = DP * MBS

LOGS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Environment variables
env = ' && '.join([
    f'cd {MINDSPEED_LLM_DIR}',
    'export CUDA_DEVICE_MAX_CONNECTIONS=1',
    'export PYTORCH_NPU_ALLOC_CONF=expandable_segments:True',
])

# Distributed torchrun arguments
distributed = ' '.join([
    'torchrun',
    f'--nproc_per_node {nproc}',
    '--nnodes 1 --node_rank 0',
    '--master_addr localhost --master_port 6000',
])

# Model architecture
model_args = ' '.join([
    '--use-mcore-models',
    '--spec mindspeed_llm.tasks.models.spec.qwen3_spec layer_spec',
    '--kv-channels 128 --qk-layernorm',
    f'--tensor-model-parallel-size {TP}',
    f'--pipeline-model-parallel-size {PP}',
    '--sequence-parallel --use-distributed-optimizer --use-flash-attn',
    '--num-layers 36 --hidden-size 4096 --num-attention-heads 32',
    '--ffn-hidden-size 12288 --max-position-embeddings 32768',
    f'--seq-length {SEQ_LENGTH}',
    '--make-vocab-size-divisible-by 1 --padded-vocab-size 151936',
    '--rotary-base 1000000 --use-rotary-position-embeddings',
])

# Training hyperparameters
train_args = ' '.join([
    f'--micro-batch-size {MBS} --global-batch-size {GBS}',
    '--disable-bias-linear --swiglu',
    f'--train-iters {TRAIN_ITERS}',
    '--tokenizer-type PretrainedFromHF',
    f'--tokenizer-name-or-path {HF_MODEL_DIR}',
    '--normalization RMSNorm --position-embedding-type rope',
    '--norm-epsilon 1e-6 --hidden-dropout 0 --attention-dropout 0',
    '--no-gradient-accumulation-fusion --attention-softmax-in-fp32',
    '--exit-on-missing-checkpoint --no-masked-softmax-fusion',
    '--group-query-attention --untie-embeddings-and-output-weights',
    '--num-query-groups 8',
    f'--min-lr {MIN_LR} --lr {LR}',
    '--weight-decay 1e-1 --clip-grad 1.0',
    '--adam-beta1 0.9 --adam-beta2 0.95 --initial-loss-scale 4096',
    '--no-load-optim --no-load-rng --seed 42 --bf16',
])

# Data and outputs
data_args = ' '.join([
    f'--data-path {PROCESSED_DATA_PREFIX}',
    '--split 100,0,0',
    '--log-interval 1',
    f'--save-interval {TRAIN_ITERS}',
    f'--eval-interval {TRAIN_ITERS} --eval-iters 0',
])

# Fine-tuning configuration
tune_args = ' '.join([
    '--finetune --stage sft --is-instruction-dataset',
    '--prompt-type qwen3 --no-pad-to-seq-lengths',
    '--distributed-backend nccl',
    f'--load {MCORE_WEIGHTS_DIR} --save {OUTPUT_DIR}',
    '--transformer-impl local',
    '--no-save-optim --no-save-rng',
])

cmd = f'{env} && {distributed} posttrain_gpt.py {model_args} {train_args} {data_args} {tune_args}'

print(f'Training configuration: {nproc} NPU, TP={TP}, PP={PP}, DP={DP}')
print(f'GBS={GBS}, MBS={MBS}, SEQ={SEQ_LENGTH}, ITERS={TRAIN_ITERS}')
print(f'\nStarting training...\n')
run_cmd(cmd, cwd=MINDSPEED_LLM_DIR)
print(f'\nTraining completed! Weights saved to: {OUTPUT_DIR}')

## 7. Inference Validation

Load the fine-tuned weights and run a generation test.

In [ ]:
import os

nproc = torch.npu.device_count()

env = ' && '.join([
    f'cd {MINDSPEED_LLM_DIR}',
    'export CUDA_DEVICE_MAX_CONNECTIONS=1',
])

distributed = ' '.join([
    'torchrun',
    f'--nproc_per_node {nproc}',
    '--nnodes 1 --node_rank 0',
    '--master_addr localhost --master_port 6001',
])

infer_args = ' '.join([
    '--use-mcore-models',
    '--spec mindspeed_llm.tasks.models.spec.qwen3_spec layer_spec',
    '--qk-layernorm',
    f'--tensor-model-parallel-size {TP}',
    f'--pipeline-model-parallel-size {PP}',
    '--num-layers 36 --hidden-size 4096 --num-attention-heads 32',
    '--ffn-hidden-size 12288',
    f'--max-position-embeddings {SEQ_LENGTH} --seq-length {SEQ_LENGTH}',
    '--disable-bias-linear',
    '--group-query-attention --num-query-groups 8',
    '--swiglu --use-fused-swiglu',
    '--normalization RMSNorm --norm-epsilon 1e-6 --use-fused-rmsnorm',
    '--position-embedding-type rope --rotary-base 1000000 --use-fused-rotary-pos-emb',
    '--make-vocab-size-divisible-by 1 --padded-vocab-size 151936',
    '--micro-batch-size 1 --max-new-tokens 256',
    '--tokenizer-type PretrainedFromHF',
    f'--tokenizer-name-or-path {HF_MODEL_DIR}',
    '--tokenizer-not-use-fast',
    '--hidden-dropout 0 --attention-dropout 0',
    '--untie-embeddings-and-output-weights',
    '--no-gradient-accumulation-fusion --attention-softmax-in-fp32',
    '--seed 42',
    f'--load {OUTPUT_DIR}',
    '--exit-on-missing-checkpoint --transformer-impl local',
])

cmd = f'{env} && {distributed} inference.py {infer_args}'
full_cmd = f'source {CANN_ENV} && source {ATB_ENV} && {cmd}'

print('Starting inference...\n')
run_env = os.environ.copy()
run_env['PYTHONWARNINGS'] = _SUPPRESS_WARNINGS
result = subprocess.run(
    ['bash', '-lc', full_cmd],
    cwd=str(MINDSPEED_LLM_DIR),
    text=True,
    input='q\n',   # Exit interactive chat mode automatically after inference.py finishes the default 4 generation rounds and enters input(); sending q terminates it
    env=run_env,
)
if result.returncode != 0:
    print(f'\nInference return code: {result.returncode}')
print('\nInference completed!')

## Using a Real Dataset

After verification succeeds, use the following steps for full fine-tuning with a real dataset:

1. **Prepare the data**: place an Alpaca/ShareGPT/Pairwise dataset inside the container
   - Alpaca: `{"instruction": "...", "input": "...", "output": "..."}`
   - Change `HANDLER_NAME` to the matching handler

2. **Tune the parameters**:
   - `SEQ_LENGTH = 4096` to match the model context length
   - `TRAIN_ITERS = 2000+` adjusted to the dataset size
   - `GBS` adjusted to the NPU count and dataset size

3. **Checkpoint interval**: change `--save-interval` in the training cell to save checkpoints periodically

4. **enable-thinking**:
   - `true` to process all data with slow-thinking mode
   - `false` to process all data with fast-thinking mode
   - `none` to mix fast and slow thinking (default)